# NASA RAG Chat Project

> **Historical exploratory notebook.** The saved outputs below use an earlier
> 2,766-chunk configuration and must not be treated as the current verified
> baseline. The canonical implementation is in `nasa_text_cleaners.py`,
> `embedding_pipeline.py`, `rag_client.py`, `llm_client.py`,
> `ragas_evaluator.py`, and `batch_evaluation.py`. Current verified results
> are stored in `evaluation_results_2026-07-28.json`.

## Source Data Loading and Cleaning

The `data_text` directory contains text extracted from NASA technical reports
and mission transcripts. Although the files are stored as plain text, many
originate from low-quality OCR or documents with complex page layouts.

The source material includes repeated headers, page markers, dates, tape
labels, incomplete pages, tables, figures, captions, and incorrectly recognized
characters. These artifacts can reduce retrieval quality if they are embedded
as though they were meaningful narrative text.

The custom `nasa_text_cleaners` module applies source-specific cleaning rules
and returns three DataFrames:

- `reports`: cleaned technical-report paragraphs;
- `transcripts`: cleaned mission transcript utterances;
- `all_data`: the combined cleaned corpus.

Reports and transcripts remain separate during chunking because they have
different structures and require different chunking strategies.

In [1]:
import pandas as pd
from collections import defaultdict
from llama_index.core.node_parser import SentenceSplitter
from nasa_text_cleaners import build_all_nasa_dataframes

CHUNK_CONFIG = {
    "report": {
        "chunk_size": 768,
        "chunk_overlap": 120,
        "paragraph_separator": "\n\n",
    },
    "transcript": {
        "chunk_size": 512,
        "chunk_overlap": 80,
        "paragraph_separator": "\n",
    },
}

CHROMA_CONFIG = {
    "directory": "chroma_db/",
    "collection_name": "nasa_collection",
    "batch_size": 100,
    "update_mode": "skip",
}

reports, transcripts, all_data = build_all_nasa_dataframes('data_text')
new_corpus = defaultdict(dict)
separator = ' '

In [2]:
for source_df in [reports, transcripts]:
    source_df = source_df.copy()
    meta_cols = pd.json_normalize(source_df["metadata"])
    source_df = pd.concat(
        [
            source_df.drop(columns=["metadata"]),
            meta_cols
        ],
        axis=1
    )
    for filepath, group in source_df.groupby('source_path'):
        source_type = group['source_type'].unique().item()
        mission = group['collection'].unique().item()
        if source_type not in CHUNK_CONFIG:
            raise ValueError(
                'The source type must be either "report" or "transcript".'
            )
        config = CHUNK_CONFIG[source_type]
        sentence_splitter = SentenceSplitter(
            chunk_size=config["chunk_size"], # CLI flag
            chunk_overlap=config["chunk_overlap"], #CLI flag
            paragraph_separator=config["paragraph_separator"],
            separator=separator,
        )
        full_text = config["paragraph_separator"].join(
            group["document"].dropna().tolist()
        )
        chunks = sentence_splitter.split_text(full_text)
        metadata = {
            'mission': mission,
            'filepath': filepath,
            'filetype': source_type
        }
        new_corpus[filepath] = {
            'chunks': chunks,
            'metadata': metadata
        }

corpus_df = (
    pd.DataFrame.from_dict(new_corpus, orient='index')
    .reset_index(names='filepath')
)
corpus_df = corpus_df.explode('chunks').reset_index(drop=True)
corpus_df["chunk_index"] = (
    corpus_df.groupby("filepath").cumcount()
)
corpus_df["doc_id"] = (
    corpus_df["filepath"].astype(str)
    + "::chunk_"
    + corpus_df["chunk_index"].astype(str)
)

## Chunk-Size Validation

Before generating embeddings, the chunks are validated using the same default
tokenizer used by `SentenceSplitter`.

For each chunk, the pipeline compares its token count with the configured
`chunk_size` for its source type. The assertion stops execution if any report
or transcript chunk exceeds its configured limit.

The descriptive statistics provide visible evidence of the resulting chunk-size
distribution for reports and transcripts.

In [3]:
from llama_index.core.utils import get_tokenizer

tokenizer = get_tokenizer()

corpus_df["token_count"] = corpus_df["chunks"].apply(
    lambda chunk: len(tokenizer(chunk))
)

corpus_df["configured_chunk_size"] = corpus_df["metadata"].apply(
    lambda metadata: CHUNK_CONFIG[
        metadata["filetype"]
    ]["chunk_size"]
)

assert (
    corpus_df["token_count"]
    <= corpus_df["configured_chunk_size"]
).all(), "At least one chunk exceeds its configured chunk size."

## OpenAI Embeddings and Persistent ChromaDB Storage

ChromaDB is used as the persistent vector store for the cleaned and chunked
NASA corpus.

The collection is configured with OpenAI's `text-embedding-3-small` model.
When chunks are added or updated, ChromaDB uses the configured embedding
function to transform each chunk into a vector representation.

A persistent ChromaDB client stores the collection under the configured
directory. The collection therefore remains available after the notebook or
Python session is restarted.

The ChromaDB directory, collection name, batch size, and update mode are exposed
through `CHROMA_CONFIG`.

In [4]:
import os
import chromadb
from tqdm import tqdm
from chromadb.config import Settings
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

chromadb_path = CHROMA_CONFIG["directory"]
collection_name = CHROMA_CONFIG["collection_name"]

client = chromadb.PersistentClient(
    path=chromadb_path,
    settings=Settings(anonymized_telemetry=False),
)

collection = client.get_or_create_collection(
    name=collection_name,
    embedding_function=OpenAIEmbeddingFunction(
        api_key=os.getenv("OPENAI_API_KEY"),
        api_base=os.getenv("OPENAI_BASE_URL"),
        model_name="text-embedding-3-small",
    ),
)

def load_chroma(
    collection,
    corpus_df,
    update_mode,
    batch_size
):
    update_modes = {
        'skip',
        'replace',
        'update'
    }
    if update_mode not in update_modes:
        raise ValueError(
            'update mode must be one of the three: "skip", "replace" or "update"'
        )
    existing_ids = set(collection.get()['ids'])
    existing_df = corpus_df[
        corpus_df['doc_id'].isin(existing_ids)
    ]
    if update_mode in {'replace', 'update'}:
        for start in tqdm(
            range(0, len(existing_df), batch_size),
            desc='modifying chroma_db'
        ):
            batch_df = existing_df.iloc[
                start: start + batch_size
            ]
            ids = batch_df["doc_id"].astype(str).tolist()
            documents = batch_df["chunks"].tolist()
            metadatas = batch_df["metadata"].tolist()
            if update_mode == 'replace':
                collection.delete(ids=ids)
                collection.add(
                    ids=ids,
                    documents=documents,
                    metadatas=metadatas
                )
            else:
                collection.update(
                    ids=ids,
                    documents=documents,
                    metadatas=metadatas
                )
    new_df = corpus_df[~corpus_df['doc_id'].isin(existing_ids)]
    for start in tqdm(
        range(0, len(new_df), batch_size), 
        desc='loading to chromadb'
    ):
        batch_df = new_df.iloc[start: start + batch_size]
        collection.add(
            ids=batch_df['doc_id'].tolist(), 
            documents=batch_df['chunks'].tolist(), 
            metadatas=batch_df['metadata'].tolist()
        )
    print(
        f"Number of records in the collection: "
        f"{collection.count()}"
    )

    print(
        f"Number of source files in the corpus: "
        f"{corpus_df['filepath'].nunique()}"
    )
    return collection


collection = load_chroma(
    collection=collection,
    corpus_df=corpus_df,
    update_mode=CHROMA_CONFIG["update_mode"],
    batch_size=CHROMA_CONFIG["batch_size"],
)

loading to chromadb: 100%|██████████| 28/28 [02:02<00:00,  4.36s/it]

Number of records in the collection: 2766
Number of source files in the corpus: 12


In [5]:
print(f"ChromaDB directory: {chromadb_path}")
print(f"Collection name: {collection.name}")
print(f"Stored chunks: {collection.count()}")
print(f"Source files: {corpus_df['filepath'].nunique()}")

mission_count = corpus_df["metadata"].apply(
    lambda metadata: metadata["mission"]
).nunique()

print(f"Missions: {mission_count}")

ChromaDB directory: chroma_db/
Collection name: nasa_collection
Stored chunks: 2766
Source files: 12
Missions: 3


# Retrieval-Augmented Conversation Pipeline

This section implements both single-turn and multi-turn question answering over
the persistent NASA ChromaDB collection.

The conversation pipeline is designed to satisfy the project requirements for
mission-specific retrieval, configurable retrieval depth, grounded answer
generation, and conversational follow-up handling.

## OpenAI Client

An OpenAI client is initialized using environment variables for the API key and
base URL. The language model is passed to each function as a runtime parameter,
allowing the model to be changed without modifying the conversation logic.

## Single-Round Retrieval and Response Generation

The `single_round_converse` function performs one complete RAG interaction.

For each user question, it:

1. selects either the original question or a rewritten standalone retrieval
   query;
2. searches the ChromaDB collection for the top `k` most relevant chunks;
3. filters retrieval by the selected NASA mission using the chunk metadata;
4. includes each retrieved chunk's rank, distance, and mission in the prompt;
5. sends the retrieved evidence and original user question to the OpenAI chat
   model;
6. returns both the generated answer and the retrieved contexts.

The values of `mission`, `k`, and `model` are provided at runtime. This makes
the retrieval scope, number of retrieved chunks, and generation model
configurable.

The system prompt requires the model to:

- answer using only the retrieved documents;
- cite factual statements using `[DOCUMENT N]`;
- avoid external knowledge and unsupported inference;
- return `Sorry, I don't know.` when the retrieved documents do not contain
  enough information;
- treat conversation history only as context for interpreting follow-up
  questions, not as factual evidence.

Keeping the retrieved contexts in the function output also allows the same
contexts to be passed directly into downstream RAGAS evaluation.

## Conversation-History Summarization

Long conversation histories can make prompts increasingly expensive and may
distract the model from the current question.

The `summarize_history` function compresses older conversation turns while
preserving information needed to interpret later follow-up questions,
including:

- NASA mission names;
- people and spacecraft;
- previously discussed events;
- unresolved references or ambiguities;
- the current subject of the conversation.

The summary is used only to resolve conversational references. Previous
assistant answers are not treated as authoritative evidence.

## Standalone Retrieval-Query Rewriting

Follow-up questions may contain pronouns or omitted subjects, such as:

`What happened after that?`

Searching the vector database with this sentence alone may retrieve irrelevant
documents because the referenced event is not explicitly named.

The `rewrite_retrieval_query` function converts the latest question into a
standalone search query using the recent conversation context. It preserves the
user's intent but does not answer the question or introduce information that is
not present in the conversation.

The rewritten query is used only for retrieval. The original user question is
still passed to the answer-generation prompt.

This separation preserves the natural conversational wording while improving
the semantic clarity of the vector search.

## Multi-Round Conversation Management

The `multi_round_converse` function coordinates conversational memory,
retrieval-query rewriting, and answer generation.

It divides the conversation history into:

- recent turns retained verbatim;
- older turns summarized into a compact conversation summary.

The number of recent verbatim messages is controlled at runtime through
`n_recent_verbatim`.

For each new question, the function:

1. copies the existing history so the original input is not modified in place;
2. summarizes older messages when the history exceeds the configured recent
   window;
3. combines the summary with the most recent verbatim turns;
4. rewrites the question as a standalone retrieval query when conversational
   context exists;
5. retrieves mission-filtered evidence from ChromaDB;
6. generates a document-grounded answer;
7. appends the new user and assistant messages to the working history;
8. returns the answer, retrieved contexts, updated history, and rewritten
   retrieval query.

Returning the rewritten query makes the retrieval process inspectable and
allows comparison between the user's conversational question and the query
actually submitted to the vector database.

## Rubric Alignment

This implementation demonstrates:

- retrieval from the configured ChromaDB collection;
- metadata filtering by NASA mission;
- configurable top-`k` retrieval;
- use of an OpenAI chat model for answer generation;
- document-grounded prompting with an explicit insufficient-evidence response;
- support for both single-round and multi-round conversations;
- follow-up-question handling through query rewriting;
- bounded conversation memory through summarization and a recent-message
  window;
- return of retrieved contexts for inspection and quantitative evaluation.

In [6]:
import os
from textwrap import dedent
from openai import OpenAI

openai_client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL")
)

def single_round_converse(openai_client, collection, user_question, history, mission, k, model, retrieval_query=None):
    query_for_retrieval = retrieval_query or user_question
    results = collection.query(
        query_texts=[query_for_retrieval],
        n_results=k,
        where={'mission': mission}
    )
    blocks = []
    for i, (chunk, distance, metadata) in enumerate(zip(results['documents'][0], results['distances'][0], results['metadatas'][0]), start=1):
        block = (
            f"[DOCUMENT {i}]\n"
            f"RETRIEVAL DISTANCE = {distance:.4f}\n"
            f"MISSION = {metadata.get('mission', 'Unknown')}\n\n"
            f"{chunk}"
        )
        blocks.append(block)
    blocks = '\n\n---\n\n'.join(blocks)
    if history:
        history_context = "RECENT CONVERSATION CONTEXT:\n" +  "\n".join(history)
    else:
        history_context = ''
    system_prompt = (
        "You are a NASA mission operations specialist. "
        "You are answering questions about some of NASA's most historic space missions. "
        "You are answering questions using ONLY information from the provided documents. "
        "Cite every factual claim using [DOCUMENT N]. "
        "Do not use Markdown formatting. Return plain text only. "
        "CRITICAL RULES: "
        "1. Every fact in your answer MUST come directly from the documents below "
        "2. If the documents don't contain enough information to answer the question, you MUST say \"Sorry, I don't know.\" "
        "3. DO NOT use any knowledge from your training data "
        "4. DO NOT make inferences beyond what is explicitly stated "
        "5. DO NOT fill in gaps with plausible-sounding information "
        "Provide your answer following the rules above. "
        "The most recent conversation context was provided for your reference. "
        "Use recent conversation context only to understand the user's follow-up question. "
        "Do not treat conversation history as factual evidence. "
    )
    user_prompt = dedent(
        history_context + 
        f"""
        USER QUESTION:
        {user_question}

        RETRIEVED DOCUMENTS:
        {blocks}
        """
    ).strip()
    response = openai_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "developer", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
    )
    answer = response.choices[0].message.content
    context = results['documents'][0]
    return answer, context


def summarize_history(openai_client, older_history, model):
    summarize_system_prompt = (
        "Summarize the earlier conversation so that a later model can "
        "understand follow-up questions that may contain pronouns, omitted "
        "subjects, or references to previously discussed topics. "
        "Preserve important named entities, NASA missions, people, spacecraft, "
        "events, the user's questions, and the current topic of discussion. "
        "Preserve unresolved questions or ambiguities. "
        "Do not predict what the user will ask next. "
        "Do not add facts that are not present in the conversation. "
        "Do not treat previous assistant answers as authoritative evidence. "
        "Return a concise plain-text summary."
    )
    prompt = '\n'.join(older_history)
    response = openai_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "developer", "content": summarize_system_prompt},
            {"role": "user", "content": prompt}
        ],
    )
    conversation_summary = response.choices[0].message.content
    return conversation_summary


def rewrite_retrieval_query(openai_client, user_question, history_for_prompt, model):
    rewrite_system_prompt = (
        "Rewrite the latest user question as a standalone retrieval query. "
        "Use the conversation history only to resolve pronouns, omitted subjects, "
        "and references to previously discussed people, missions, spacecraft, "
        "events, or topics. "
        "Preserve the user's original intent. "
        "Do not answer the question. "
        "Do not add information that is not present in the question or history. "
        "Return only the rewritten query."
    )
    prompt = (
        'CONVERSATION CONTEXT:\n'
        + '\n'.join(history_for_prompt)
        + f'\n\nLATEST USER QUESTION:\n{user_question}'
    )
    response = openai_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "developer", "content": rewrite_system_prompt},
            {"role": "user", "content": prompt}
        ],
    )
    retrieval_query = response.choices[0].message.content.strip()
    return retrieval_query


def multi_round_converse(openai_client, collection, user_question, history, mission, k, n_recent_verbatim, model):
    working_history = list(history)
    older_history = working_history[:-n_recent_verbatim]
    recent_history = working_history[-n_recent_verbatim:]
    retrieval_query = None
    if len(working_history) <= n_recent_verbatim:
        history_for_prompt = working_history
    else:
        conversation_summary = summarize_history(openai_client, older_history, model)
        history_for_prompt = [
            f'CONVERSATION SUMMARY: {conversation_summary}',
            *recent_history
        ]
    if len(history_for_prompt) > 0:
        retrieval_query = rewrite_retrieval_query(
            openai_client,
            user_question=user_question,
            history_for_prompt=history_for_prompt,
            model=model
        )
    answer, context = single_round_converse(
        openai_client,
        collection,
        user_question,
        history=history_for_prompt,
        k=k,
        mission=mission,
        model=model,
        retrieval_query=retrieval_query
    )
    working_history.extend([
        f'USER: {user_question}',
        f'ASSISTANT: {answer}'
    ])
    return answer, context, working_history, retrieval_query


# RAGAS Evaluation Configuration

This section configures the language-model judge, embedding model, and RAGAS
metrics used to evaluate the NASA retrieval-augmented generation pipeline.

The evaluation components are kept separate from the answer-generation
pipeline. The RAG system first generates an answer and returns the retrieved
contexts. RAGAS then evaluates those outputs using the user question, retrieved
documents, generated response, and reference answer as required by each metric.

## Asynchronous Evaluation Client

An `AsyncOpenAI` client is used for evaluation so that multiple independent
metrics can be scored concurrently in a later stage of the notebook.

The API key and base URL are read from environment variables rather than being
stored directly in the notebook.

The judge model is created with `llm_factory` and supplied to the RAGAS
metrics. The model name is configurable, allowing a different judge model to
be selected without changing the evaluation logic.

A maximum output budget of 8,192 tokens is provided because some metrics,
particularly factual-correctness evaluation, may need to:

1. decompose an answer into individual factual claims;
2. compare those claims with the reference answer;
3. return a structured verification result.

The larger output allowance reduces the risk that the structured judge response
is truncated for questions containing many claims.

## Evaluation Embeddings

The evaluation pipeline uses OpenAI's `text-embedding-3-small` model through
RAGAS's embedding factory.

Evaluation embeddings are required by metrics that compare the semantic meaning
of generated and expected content. They are separate from the embeddings
already stored in ChromaDB, even though the same embedding model may be used.

## Selected RAGAS Metrics

The `build_ragas_scorers` function constructs the metrics used for every test
sample.

### Faithfulness

`Faithfulness` measures whether factual claims in the generated response are
supported by the retrieved contexts.

It evaluates grounding rather than whether the answer matches the reference
answer. A response can therefore be faithful to its retrieved documents while
still being incomplete or irrelevant to the question.

Required evaluation inputs:

- generated response;
- retrieved contexts.

### Answer Relevancy

`AnswerRelevancy` measures how directly the generated response addresses the
user's question.

It uses both the judge language model and evaluation embeddings. A response may
be factually supported but receive a low relevancy score when it is incomplete,
off-topic, or does not directly answer the question.

Required evaluation inputs:

- user question;
- generated response.

### Context Precision

`ContextPrecision` evaluates whether the retrieved contexts are relevant and
useful for producing the reference answer.

A high score indicates that the retriever returned useful evidence without
placing large amounts of irrelevant material among the retrieved chunks.

Required evaluation inputs:

- user question;
- reference answer;
- retrieved contexts.

### Context Recall

`ContextRecall` evaluates whether the retrieved contexts contain the information
needed to support the reference answer.

A low score indicates that important evidence was not retrieved, even when some
of the returned chunks were relevant.

Required evaluation inputs:

- user question;
- reference answer;
- retrieved contexts.

### Factual Correctness

`FactualCorrectness` compares factual claims in the generated response with the
reference answer.

It is configured with `mode="f1"`, which balances:

- factual precision: how many claims in the generated response are supported by
  the reference;
- factual recall: how much of the reference information is covered by the
  generated response.

This metric evaluates agreement with the expected answer rather than grounding
against the retrieved documents.

Required evaluation inputs:

- generated response;
- reference answer.

## Metric Separation

The selected metrics evaluate different parts of the RAG pipeline:

| Pipeline component | Metric |
|---|---|
| Grounding of the generated answer | Faithfulness |
| Responsiveness to the question | Answer Relevancy |
| Relevance of retrieved evidence | Context Precision |
| Completeness of retrieved evidence | Context Recall |
| Agreement with the reference answer | Factual Correctness |

Keeping these dimensions separate makes failure analysis more useful.

For example:

- high faithfulness with low factual correctness may indicate that the answer
  accurately reflects incomplete or incorrect retrieval;
- high context precision with low context recall may indicate that the returned
  chunks are relevant but important evidence is missing;
- strong retrieval scores with low answer relevancy may indicate a
  generation or prompting problem rather than a retrieval problem.

## Optional Answer Correctness Metric

`AnswerCorrectness` is included in the code as an optional commented-out
metric. It combines factual comparison with semantic similarity.

It is not enabled in the initial evaluation because its components partially
overlap with the separately reported factual-correctness and answer-relevancy
dimensions. Reporting the individual metrics first makes the source of poor
performance easier to diagnose.

## Runtime Scorer Construction

The scorer objects are created through `build_ragas_scorers` rather than as
global hardcoded metric instances.

This makes the evaluation configuration reusable and allows the judge model or
embedding model to be replaced at runtime while preserving the same metric
definitions.

The resulting `scorers` dictionary provides a consistent mapping between metric
names and RAGAS scorer objects for the batch-evaluation pipeline.

In [7]:
import os
import asyncio
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.embeddings.base import embedding_factory
from ragas.metrics.collections import (
    Faithfulness,
    AnswerRelevancy,
    ContextPrecision,
    ContextRecall,
    FactualCorrectness,
    # AnswerCorrectness
)

judge_client = AsyncOpenAI(
    api_key=os.environ["OPENAI_API_KEY"],
    base_url=os.environ["OPENAI_BASE_URL"],
)

judge_llm = llm_factory(
    "gpt-5-nano",
    client=judge_client,
    max_tokens=8192
)

judge_embeddings = embedding_factory(
    'openai',
    model='text-embedding-3-small',
    client=judge_client
)

def build_ragas_scorers(*, judge_llm, judge_embeddings):
    return {
        'faithfulness': Faithfulness(
            llm=judge_llm
        ),
        'answer_relevancy': AnswerRelevancy(
            llm=judge_llm,
            embeddings=judge_embeddings
        ),
        'context_precision': ContextPrecision(
            llm=judge_llm
        ),
        'context_recall': ContextRecall(
            llm=judge_llm
        ),
        'factual_correctness': FactualCorrectness(
            llm=judge_llm,
            mode='f1'
        ),
        # 'answer_correctness': AnswerCorrectness(
        #     llm=judge_llm,
        #     embeddings=judge_embeddings,
        #     weights=[0.75, 0.25],
        #     beta=1.0
        # )
    }

scorers = build_ragas_scorers(judge_llm=judge_llm, judge_embeddings=judge_embeddings)

# Asynchronous RAG Evaluation Pipeline

This section connects the RAG response-generation pipeline to the configured
RAGAS scorers.

For each test question, the pipeline:

1. retrieves mission-specific contexts from ChromaDB;
2. generates a grounded answer using the selected generator model;
3. constructs the evaluation sample required by RAGAS;
4. evaluates all configured metrics concurrently;
5. records metric scores, execution times, and errors;
6. returns one structured result row for later aggregation and reporting.

## Metric Input Mapping

Different RAGAS metrics require different combinations of evaluation fields.
The `METRIC_FIELDS` dictionary explicitly maps every enabled metric to the
fields it needs.

The available fields are:

- `user_input`: the original test question;
- `response`: the answer generated by the RAG system;
- `reference`: the expected answer from the evaluation dataset;
- `retrieved_contexts`: the document chunks returned by ChromaDB.

The mappings reflect the different purposes of the metrics:

| Metric | Required fields | Evaluation purpose |
|---|---|---|
| Faithfulness | Question, response, retrieved contexts | Determines whether the response is supported by retrieved evidence |
| Answer Relevancy | Question, response | Determines whether the response directly addresses the question |
| Context Precision | Question, reference, retrieved contexts | Determines whether retrieved chunks are relevant and well ranked |
| Context Recall | Question, reference, retrieved contexts | Determines whether the retrieval contains the evidence needed for the reference answer |
| Factual Correctness | Response, reference | Compares the factual claims in the generated and expected answers |

Centralizing these mappings avoids duplicating metric-specific argument logic
throughout the evaluation code and makes additional metrics easier to add.

## Individual Metric Evaluation

The asynchronous `score_metric` function evaluates one metric for one test
sample.

It performs the following operations:

1. looks up the fields required by the metric;
2. extracts those fields from the evaluation sample;
3. records the metric start time;
4. calls the scorer's asynchronous `ascore` method;
5. returns the score and elapsed time.

The returned result contains:

- the metric name;
- the numeric score;
- the metric execution time;
- an error message when evaluation fails.

## Per-Metric Error Handling

Each metric is wrapped in its own `try`/`except` block.

If a scorer fails, its score is recorded as `NaN`, while the exception type and
message are stored in a dedicated error field. Other metrics can still complete
and their results remain available.

For example, a failed factual-correctness evaluation produces fields similar
to:

`factual_correctness = NaN`

`factual_correctness_error = IncompleteOutputException: ...`

This behavior is important because language-model-based evaluation may
occasionally fail because of output truncation, structured-output parsing, or
temporary API errors. A failure in one metric should not discard the complete
evaluation result for that question.

## Concurrent Metric Execution

The `score_all` function creates one coroutine for each configured scorer and
executes them concurrently using `asyncio.gather`.

The metrics are independent once the generated answer and retrieved contexts
are available, so running them concurrently reduces total evaluation time
compared with evaluating them sequentially.

The function returns three dictionaries:

- `scores`: one numeric value for each metric;
- `timings`: one execution-time value for each metric;
- `errors`: one optional error message for each metric.

Each timing and error field uses a metric-specific suffix, such as:

- `faithfulness_seconds`;
- `faithfulness_error`;
- `context_recall_seconds`;
- `context_recall_error`.

Because the metrics run concurrently, `evaluation_seconds` represents the
wall-clock duration of the complete concurrent evaluation. It is not expected
to equal the sum of the individual metric durations.

## Evaluating One Question–Answer Sample

The `evaluate_one_qa` function performs the complete generation and evaluation
workflow for one item from the test dataset.

### Response generation

The function first calls `single_round_converse` with:

- the test question;
- its associated NASA mission;
- the configured retrieval depth, `top_k`;
- the selected generator model;
- an empty conversation history.

This produces:

- the generated answer;
- the exact contexts retrieved from ChromaDB.

Using the same retrieved contexts for answer generation and evaluation ensures
that faithfulness and retrieval metrics assess the evidence actually supplied
to the model.

### Evaluation-sample construction

The original test record is extended with:

- `response`;
- `retrieved_contexts`.

The resulting sample contains all fields required by the configured RAGAS
metrics.

### Timing

Three levels of elapsed time are recorded:

- `generation_seconds`: retrieval and answer-generation time;
- `evaluation_seconds`: concurrent RAGAS evaluation time;
- `total_seconds`: complete generation-and-evaluation time.

Individual metric times are also retained for performance inspection.

### Structured output

The function returns one dictionary representing one evaluation row. It
contains:

- the original test-question fields;
- the generated response;
- the retrieved contexts;
- the number of retrieved contexts;
- overall and per-metric timings;
- all metric scores;
- all metric error messages.

This one-question-to-one-row structure can be converted directly into a Pandas
DataFrame and used for checkpointing, aggregation, failure analysis, and final
reporting.

## Rubric Alignment

This evaluation pipeline demonstrates:

- automated evaluation over a structured question-and-reference dataset;
- use of the actual RAG-generated response and retrieved contexts;
- separate evaluation of grounding, relevance, retrieval quality, and factual
  correctness;
- configurable generator model and retrieval depth;
- asynchronous execution of independent evaluation metrics;
- per-question and per-metric timing;
- fault tolerance when an individual judge call fails;
- structured outputs suitable for aggregation and quantitative reporting.

The judge client should be closed after all asynchronous evaluations have
completed:

`await judge_client.close()`

In [ ]:
import asyncio
import math


METRIC_FIELDS = {
    'faithfulness': (
        'user_input',
        'response',
        'retrieved_contexts'
    ),
    'answer_relevancy': (
        'user_input',
        'response'
    ),
    'context_precision': (
        'user_input',
        'reference',
        'retrieved_contexts'
    ),
    'context_recall': (
        'user_input',
        'reference',
        'retrieved_contexts'
    ),
    'factual_correctness': (
        'response',
        'reference'
    ),
    # 'answer_correctness': (
    #     'user_input',
    #     'reference'
    # )
}

async def score_metric(*, metric_name, scorer, sample):
    required_fields = METRIC_FIELDS[metric_name]
    kwargs = {field: sample[field] for field in required_fields}
    start = perf_counter()
    try:
        result = await scorer.ascore(**kwargs)
        return {
            'name': metric_name,
            'score': float(result.value),
            'seconds': perf_counter() - start,
            'error': None
        }
    except Exception as exc:
        return {
            'name': metric_name,
            'score': math.nan,
            'seconds': perf_counter() - start,
            'error': f"{type(exc).__name__}: {exc}"
        }



async def score_all(*, scorers, sample):
    coroutines = [
        score_metric(
            metric_name=metric_name,
            scorer=scorer,
            sample=sample
        ) for metric_name, scorer in scorers.items()
    ]
    metric_results = await asyncio.gather(
        *coroutines
    )
    scores = {
        result['name']: result['score']
        for result in metric_results
    }
    timings = {
        f'{result["name"]}_seconds': result['seconds']
        for result in metric_results
    }
    errors = {
        f'{result["name"]}_error': result['error']
        for result in metric_results
    }
    return scores, timings, errors


async def evaluate_one_qa(*, qa, openai_client, collection, scorers, generator_model, top_k):
    
    total_start = perf_counter()
    
    generation_start = perf_counter()
    generated_answer, retrieved_contexts = single_round_converse(
        openai_client=openai_client,
        collection=collection,
        user_question=qa['user_input'],
        history=[],
        mission=qa['mission'],
        k=top_k,
        model=generator_model,
    )
    generation_seconds = perf_counter() - generation_start
    evaluation_sample = {
        **qa,
        'response': generated_answer,
        'retrieved_contexts': retrieved_contexts,
    }

    evaluation_start = perf_counter()
    scores, timings, errors = await score_all(scorers=scorers, sample=evaluation_sample)
    evaluation_seconds = perf_counter() - evaluation_start

    total_seconds = perf_counter() - total_start
    result_row = {
        **qa,
        'response': generated_answer,
        'retrieved_contexts': retrieved_contexts,
        'context_count': len(retrieved_contexts),
        'generation_seconds': generation_seconds,
        'evaluation_seconds': evaluation_seconds,
        'total_seconds': total_seconds,
        **scores,
        **timings,
        **errors
    }
    return result_row

# await judge_client.close()

# Batch Evaluation with Checkpointing

This section runs the RAG and RAGAS evaluation pipeline over the complete test
question set.

The test questions are loaded from `test_questions.json`. Each item is expected
to contain the fields required by the evaluation pipeline, including:

- `id`: a unique identifier for the test question;
- `mission`: the NASA mission used for metadata-filtered retrieval;
- `user_input`: the question submitted to the RAG system;
- `reference`: the expected answer used by reference-based metrics;
- any additional descriptive fields, such as question category or source.

## Resumable Evaluation

Language-model-based evaluation can take several minutes and requires multiple
API calls for each question. A complete run may also be interrupted by a
notebook restart, network error, API timeout, or individual metric failure.

To avoid losing completed results, the evaluation loop uses a persistent Pandas
checkpoint file:

`checkpoint_df.pkl`

Before processing each question, the loop reloads the checkpoint and extracts
the IDs that have already been evaluated.

Questions whose IDs are present in the checkpoint are skipped. The first
remaining question is evaluated and immediately appended to the checkpoint.

This produces the following cycle:

1. load the latest checkpoint;
2. identify completed question IDs;
3. identify questions that are still pending;
4. evaluate the next pending question;
5. append its result as one DataFrame row;
6. save the updated checkpoint;
7. repeat until no pending questions remain.

Because the checkpoint is written after every question, an interrupted run can
resume without repeating the completed evaluations.

## One Question per Result Row

The dictionary returned by `evaluate_one_qa` is wrapped in a list before being
converted into a DataFrame:

`pd.DataFrame([new_row])`

The outer list indicates that the complete result dictionary represents one
row.

This is important because some fields, such as `retrieved_contexts`, contain
lists. The contexts should remain inside a single result row rather than being
expanded into multiple rows.

Each checkpoint row therefore represents one test question and contains:

- the original test-question fields;
- the generated answer;
- the retrieved contexts;
- metric scores;
- metric-specific error messages;
- generation and evaluation timings.

## Evaluation Configuration

Each pending question is evaluated using:

- generator model: `gpt-5-nano`;
- retrieval depth: `top_k=5`;
- mission-filtered retrieval from the NASA ChromaDB collection;
- the complete set of configured RAGAS scorers.

The generator model and retrieval depth are passed into
`evaluate_one_qa`, allowing them to be changed without modifying the internal
evaluation functions.

## Completion Condition

The loop terminates when every question ID from `test_questions.json` is
present in the checkpoint.

At that point, the notebook prints:

`All questions completed`

The resulting checkpoint DataFrame can then be used for metric aggregation,
category-level analysis, error inspection, visualization, and final reporting.

## Rubric Alignment

This batch-evaluation procedure demonstrates:

- automated evaluation over the full test dataset;
- one structured evaluation record per question;
- persistent storage of generated answers, contexts, and metric scores;
- recovery from interrupted notebook or API sessions;
- avoidance of duplicate evaluations through unique question IDs;
- configurable generation and retrieval settings;
- outputs suitable for aggregate quantitative reporting.

In [9]:
from time import perf_counter
import json

with open('test_questions.json') as f:
    test_questions = json.load(f)['questions']

checkpoint_df_name = 'checkpoint_df.pkl'
while True:
    try:
        checkpoint_df = pd.read_pickle(checkpoint_df_name)
    except FileNotFoundError:
        checkpoint_df = pd.DataFrame()
    if checkpoint_df.empty:
        completed_ids = set()
    else:
        completed_ids = set(checkpoint_df['id'])
    pending_questions = [
        qa
        for qa in test_questions
        if qa['id'] not in completed_ids
    ]
    if len(pending_questions) == 0:
        print('All questions completed')
        break
    next_qa = pending_questions[0]
    new_row = await evaluate_one_qa(
        qa=next_qa,
        openai_client=openai_client,
        collection=collection,
        scorers=scorers,
        generator_model="gpt-5-nano",
        top_k=5,
    )
    checkpoint_df = pd.concat(
        [checkpoint_df, pd.DataFrame([new_row])],
        ignore_index=True
    )
    checkpoint_df.to_pickle(checkpoint_df_name)
    

All questions completed


# Aggregate Evaluation Results

This section summarizes the question-level evaluation results stored in
`checkpoint_df`.

## Retrieval F1

Context precision and context recall measure complementary aspects of the
retrieval system:

- **Context precision** indicates how much of the retrieved evidence is relevant.
- **Context recall** indicates how much of the evidence required by the reference
  answer was retrieved.

A retrieval F1 score is calculated separately for each test question using the
harmonic mean of context precision and context recall.

Let \(P\) represent context precision and \(R\) represent context recall:

$$
F1_{\mathrm{retrieval}}
=
\frac{2PR}{P + R}
$$

The harmonic mean is appropriate because retrieval F1 should be high only when
both precision and recall are high. A low value in either component substantially
reduces the combined score.

When both precision and recall are zero, the formula produces an undefined
`0 / 0` result. The code replaces this result with `0`, representing complete
retrieval failure for that question.

## Metric Aggregation

The derived `retrieval_f1` column is inserted alongside the five RAGAS metrics:

- faithfulness;
- answer relevancy;
- context precision;
- context recall;
- retrieval F1;
- factual correctness.

The `describe()` method then calculates descriptive statistics across all test
questions:

- `count`: number of available results;
- `mean`: arithmetic average across questions;
- `std`: standard deviation across questions;
- `min`: lowest score;
- `25%`: first quartile;
- `50%`: median;
- `75%`: third quartile;
- `max`: highest score.

Retrieval F1 is therefore calculated at the individual-question level first and
then averaged across the test set. This is a macro-averaged retrieval F1 score.

Reporting the mean together with the median, quartiles, minimum, maximum, and
standard deviation helps reveal whether performance is consistent or whether
the overall average is being influenced by a small number of particularly strong
or weak questions.

In [10]:
summary_df = checkpoint_df.copy()
precision = summary_df["context_precision"]
recall = summary_df["context_recall"]
summary_df["retrieval_f1"] = (
    2 * precision * recall / (precision + recall)
).fillna(0)
metric_keys = list(METRIC_FIELDS.keys())
metric_keys.insert(4, 'retrieval_f1')
summary_df = summary_df.set_index('id').filter(metric_keys).describe()
summary_df[metric_keys]

,faithfulness,answer_relevancy,context_precision,context_recall,retrieval_f1,factual_correctness
count,17.000000,17.000000,17.000000,17.000000,17.000000,17.000000
mean,0.696078,0.490591,0.486356,0.519608,0.462169,0.338824
std,0.433955,0.351566,0.387072,0.399141,0.363642,0.309836
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.333333,0.000000,0.250000,0.000000,0.000000,0.000000
50%,1.000000,0.571830,0.366667,0.500000,0.473118,0.430000
75%,1.000000,0.776250,0.866667,1.000000,0.753623,0.640000
max,1.000000,0.941036,1.000000,1.000000,1.000000,0.750000


## Overall Evaluation and Conclusions

The evaluation results show that the RAG pipeline is operational and can
produce well-grounded answers for some questions, but its performance is
inconsistent across the 17-question test set.

### Grounding

Faithfulness is the strongest metric:

- mean: `0.696`;
- median: `1.000`;
- maximum: `1.000`.

The median of `1.000` indicates that at least half of the generated answers were
fully supported by the retrieved documents.

However, the minimum score of `0` and the high standard deviation of `0.434`
show that grounding is not reliable for every question. Some answers are fully
supported, while others fail completely.

### Answer Relevancy

Answer relevancy has:

- mean: `0.491`;
- median: `0.572`;
- first quartile: `0.000`.

This indicates that the generated answers only moderately address the user
questions. At least approximately one quarter of the responses received a
relevancy score of zero.

The model may therefore generate information that is grounded in the retrieved
documents but does not directly or completely answer the question.

### Retrieval Quality

Retrieval performance is moderate:

- context precision mean: `0.486`;
- context recall mean: `0.520`;
- retrieval F1 mean: `0.462`;
- retrieval F1 median: `0.473`.

Context recall is slightly higher than context precision. This suggests that
the retriever sometimes finds relevant evidence, but the top retrieved chunks
may also contain irrelevant or incomplete information.

The retrieval F1 first quartile is `0`, showing that retrieval failed
substantially for at least approximately one quarter of the test questions.

### Factual Correctness

Factual correctness is the weakest metric:

- mean: `0.339`;
- median: `0.430`;
- maximum: `0.750`.

The generated answers therefore have limited agreement with the reference
answers. Even the best evaluated response did not achieve complete factual
coverage.

This may indicate that the generated answers:

- omit important facts from the reference answer;
- include claims not supported by the reference;
- receive incomplete evidence from retrieval;
- provide only partial answers.

### Overall Assessment

The system should not yet be considered a high-quality RAG implementation based
on these results.

Its main strength is that it can generate grounded answers when relevant
evidence is successfully retrieved. Its main weaknesses are inconsistent
retrieval, incomplete answers, and low factual agreement with the reference
answers.

A suitable overall description is:

> The RAG pipeline functions correctly as an end-to-end proof of concept and
> produces strongly grounded answers for some questions. However, retrieval
> quality and factual completeness remain inconsistent, so further improvement
> is required before the system can be considered reliable.

## Possible Sources of Error

The evaluation results alone cannot prove the cause of the weak performance,
but several factors are plausible.

### Source-document quality

The NASA corpus contains OCR errors, incomplete pages, repeated headers,
tables, figures, and fragmented text. These artifacts may reduce embedding
quality and cause important evidence to be split, corrupted, or ranked below
irrelevant chunks.

Poor source quality is therefore a likely contributor to the moderate context
precision, context recall, and retrieval F1 scores.

### Generator-model capability

The pipeline uses `gpt-5-nano` as the answer-generation model. This model was
selected to reduce evaluation cost and runtime.

A smaller and less expensive model may be less capable of:

- combining information across multiple retrieved chunks;
- producing complete answers;
- following detailed citation instructions;
- distinguishing relevant evidence from distracting context.

This may contribute to the low answer-relevancy and factual-correctness scores.

### Evaluation-model limitations

The same `gpt-5-nano` model is also used as the RAGAS judge. Consequently, the
evaluation is cost-efficient but is not fully independent of the generation
model.

Some score variation may reflect limitations or instability in the judge model
rather than only weaknesses in the RAG system.

### Retrieval configuration

The current results may also depend on:

- the selected chunk sizes and overlaps;
- the fixed retrieval depth of `top_k=5`;
- the absence of a reranking stage;
- the quality of the rewritten retrieval query;
- the embedding model;
- the quality and completeness of the reference answers.

## Limitations of the Conclusion

The test set contains only 17 questions. The results are useful for diagnosing
the current implementation, but they are not sufficient to make a definitive
claim about performance across all NASA mission questions.

To identify the actual causes of failure, the pipeline should be compared under
controlled experiments, such as:

- cleaned corpus versus minimally cleaned corpus;
- `gpt-5-nano` versus a more capable generator model;
- different chunk sizes and overlaps;
- different values of `top_k`;
- vector retrieval alone versus retrieval followed by reranking.

These comparisons would help separate data-quality, retrieval, generation, and
evaluation-model effects.